#### Necessary Imports

In [1]:
import requests, json
from PIL import Image
import io, os

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 3, Finished, Available, Finished, False)

#### Credentials from Key Vault

In [2]:
'''
vault_url     = "https://my-key-resource.vault.azure.net/"
api_key       = notebookutils.credentials.getSecret(vault_url, "ai-vision-api-key")
endpoint      = notebookutils.credentials.getSecret(vault_url, "ai-vision-endpoint")
'''

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 4, Finished, Available, Finished, False)

'\nvault_url     = "https://my-key-resource.vault.azure.net/"\napi_key       = notebookutils.credentials.getSecret(vault_url, "ai-vision-api-key")\nendpoint      = notebookutils.credentials.getSecret(vault_url, "ai-vision-endpoint")\n'

In [2]:
vault_url     = "https://cv-training-key.vault.azure.net/"
api_key       = notebookutils.credentials.getSecret(vault_url, "det-obj-key")
endpoint      = notebookutils.credentials.getSecret(vault_url, "det-obj-endpoint")

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 4, Finished, Available, Finished, False)

#### Image analyzer

In [3]:
# Tagged Parameters cell
image_path = "test-object-detection/image2.jpg"  

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 5, Finished, Available, Finished, False)

In [4]:
# --- Copy image from Lakehouse to /tmp/ ---
files_listing  = notebookutils.fs.ls("Files")
lakehouse_root = files_listing[0].path.split("/Files/")[0]
abs_image_path = f"{lakehouse_root}/Files/{image_path}"
notebookutils.fs.cp(abs_image_path, "file:/tmp/input.jpg")

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 6, Finished, Available, Finished, False)

True

#### Call Azure AI Vision Object Detection AP

In [5]:
detect_url = f"{endpoint}/computervision/imageanalysis:analyze"

params  = {
    "features": "objects,tags",   # objects returns bounding boxes
    "api-version": "2024-02-01"
}
headers = {
    "Ocp-Apim-Subscription-Key": api_key,
    "Content-Type": "application/octet-stream"
}

with open("/tmp/input.jpg", "rb") as img:
    response = requests.post(detect_url, params=params, headers=headers, data=img)

result = response.json()

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 7, Finished, Available, Finished, False)

In [6]:
print(result)

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 8, Finished, Available, Finished, False)

{'modelVersion': '2023-10-01', 'metadata': {'width': 612, 'height': 360}, 'tagsResult': {'values': [{'name': 'mammal', 'confidence': 0.9823590517044067}, {'name': 'pet', 'confidence': 0.974373459815979}, {'name': 'animal', 'confidence': 0.9735958576202393}, {'name': 'dog breed', 'confidence': 0.9350380897521973}, {'name': 'fur', 'confidence': 0.8876129388809204}, {'name': 'snout', 'confidence': 0.8782972097396851}, {'name': 'whiskers', 'confidence': 0.8556392192840576}, {'name': 'dog', 'confidence': 0.8533998727798462}, {'name': 'cat', 'confidence': 0.6959854364395142}]}, 'objectsResult': {'values': [{'boundingBox': {'x': 59, 'y': 89, 'w': 160, 'h': 247}, 'tags': [{'name': 'cat', 'confidence': 0.821}]}, {'boundingBox': {'x': 393, 'y': 65, 'w': 134, 'h': 273}, 'tags': [{'name': 'dog', 'confidence': 0.787}]}, {'boundingBox': {'x': 205, 'y': 47, 'w': 253, 'h': 298}, 'tags': [{'name': 'dog', 'confidence': 0.884}]}]}}


#### Filter for animal/pet detections

In [13]:
'''
PET_TAGS  = {"dog", "cat", "animal", "pet", "kitten", "puppy", "canine", "feline"}
THRESHOLD = 0.5

# Step 1 — Check image-level tags for pet presence
image_tags    = {t["name"].lower() for t in result.get("tagsResult", {}).get("values", [])
                 if t["confidence"] >= THRESHOLD}
image_has_pet = bool(image_tags & PET_TAGS)

print(f"Image-level tags found:     {image_tags}")
print(f"Pet detected at image level: {image_has_pet}")

# Step 2 — Filter objects using ALL tags per object including parent categories
detections = []
for obj in result.get("objectsResult", {}).get("values", []):
    obj_tags       = {t["name"].lower() for t in obj.get("tags", [])}
    obj_confidence = obj.get("tags", [{}])[0].get("confidence", 0)
    is_pet         = bool(obj_tags & PET_TAGS)

    if obj_confidence >= THRESHOLD and (is_pet or image_has_pet):
        detections.append(obj)

print(f"Objects matching pet criteria: {len(detections)}")

# Step 3 — Crop or log no detection
if not detections:
    print("⚠️ No pet detected")
else:
    best = max(detections, key=lambda o: o["tags"][0]["confidence"])
    box  = best["boundingBox"]

    with Image.open("/tmp/input.jpg") as img:
        cropped = img.crop((
            box["x"],
            box["y"],
            box["x"] + box["w"],
            box["y"] + box["h"]
        ))
        cropped.save("/tmp/cropped.jpg")

    # Save cropped image permanently to Lakehouse
    filename     = os.path.basename(image_path)
    cropped_dest = f"{lakehouse_root}/Files/development/cropped/{filename}"
    notebookutils.fs.cp("file:/tmp/cropped.jpg", cropped_dest)
    print(f"✅ Pet detected and cropped → saved to: {cropped_dest}")
    '''

StatementMeta(, 8b222e13-c13a-43cd-9dbb-699e5802898a, 15, Finished, Available, Finished, False)

Image-level tags found:     {'sky', 'dog breed', 'grass', 'outdoor', 'person', 'woman', 'smile', 'dog', 'girl'}
Pet detected at image level: True
Objects matching pet criteria: 3
✅ Pet detected and cropped → saved to: abfss://a96eab2a-8002-45f2-92b4-d2bf05c2540b@onelake.dfs.fabric.microsoft.com/db748fcd-0cc5-478c-a54c-f4938d542b0f/Files/development/cropped/images.jpeg


In [10]:
from datetime import datetime
from pyspark.sql import Row

PET_TAGS  = {"dog", "cat", "animal", "pet", "kitten", "puppy", "canine", "feline"}
THRESHOLD = 0.5

# Step 1 — Image-level tags
image_tags    = {t["name"].lower() for t in result.get("tagsResult", {}).get("values", [])
                 if t["confidence"] >= THRESHOLD}
image_has_pet = bool(image_tags & PET_TAGS)

print(f"Image-level tags found:      {image_tags}")
print(f"Pet detected at image level: {image_has_pet}")

# Step 2 — Filter pet objects only
detections = []
for obj in result.get("objectsResult", {}).get("values", []):
    obj_tags       = {t["name"].lower() for t in obj.get("tags", [])}
    obj_confidence = obj.get("tags", [{}])[0].get("confidence", 0)
    is_pet         = bool(obj_tags & PET_TAGS) or (
        image_has_pet and not bool(obj_tags & {"person", "woman", "man", "girl", "boy", "child"})
    )
    if obj_confidence >= THRESHOLD and is_pet:
        detections.append(obj)

print(f"Pet objects found before deduplication: {len(detections)}")

# Step 3 — Deduplicate
def iou(box1, box2):
    x1 = max(box1["x"], box2["x"])
    y1 = max(box1["y"], box2["y"])
    x2 = min(box1["x"] + box1["w"], box2["x"] + box2["w"])
    y2 = min(box1["y"] + box1["h"], box2["y"] + box2["h"])
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    union = box1["w"]*box1["h"] + box2["w"]*box2["h"] - intersection
    return intersection / union if union > 0 else 0

def deduplicate(detections, iou_threshold=0.3):
    sorted_dets = sorted(detections, key=lambda o: o["tags"][0]["confidence"], reverse=True)
    kept = []
    for det in sorted_dets:
        if not any(iou(det["boundingBox"], k["boundingBox"]) > iou_threshold for k in kept):
            kept.append(det)
    return kept

detections     = deduplicate(detections)
filename_base  = os.path.splitext(os.path.basename(image_path))[0]
timestamp      = datetime.now().strftime("%Y%m%d_%H%M%S")
cropped_paths  = []
rows           = []

print(f"Pet objects after deduplication: {len(detections)}")

# Step 4 — Crop, save and build metrics rows together
if not detections:
    print("⚠️ No pet detected in image")
    rows.append(Row(
        image_name   = f"{filename_base}_{timestamp}_no_detection.jpg",
        detected     = False,
        animal_count = 0,
        object_name  = "none",
        confidence   = 0.0,
        bbox_x=0, bbox_y=0, bbox_w=0, bbox_h=0,
        timestamp    = datetime.now()
    ))
else:
    for i, det in enumerate(detections):
        box          = det["boundingBox"]
        tag          = det["tags"][0]["name"]
        conf         = det["tags"][0]["confidence"]
        cropped_name = f"{filename_base}_{timestamp}_{i}.jpg"

        with Image.open("/tmp/input.jpg") as img:
            cropped = img.crop((box["x"], box["y"], box["x"]+box["w"], box["y"]+box["h"]))
            local_out = f"/tmp/cropped_{i}.jpg"
            cropped.save(local_out)

        cropped_dest = f"{lakehouse_root}/Files/development/cropped/{cropped_name}"
        notebookutils.fs.cp(f"file:{local_out}", cropped_dest)
        cropped_paths.append(cropped_dest)

        rows.append(Row(
            image_name   = cropped_name,
            detected     = True,
            animal_count = len(detections),
            object_name  = tag,
            confidence   = float(conf),
            bbox_x       = int(box["x"]),
            bbox_y       = int(box["y"]),
            bbox_w       = int(box["w"]),
            bbox_h       = int(box["h"]),
            timestamp    = datetime.now()
        ))

        print(f"✅ Crop {i+1}: '{tag}' ({conf:.2%}) → {cropped_name}")

    print(f"\n📦 Total crops saved: {len(cropped_paths)}")

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 12, Finished, Available, Finished, False)

Image-level tags found:      {'cat', 'dog breed', 'mammal', 'whiskers', 'animal', 'fur', 'pet', 'snout', 'dog'}
Pet detected at image level: True
Pet objects found before deduplication: 3
Pet objects after deduplication: 3
✅ Crop 1: 'dog' (88.40%) → image2_20260502_152305_0.jpg
✅ Crop 2: 'cat' (82.10%) → image2_20260502_152305_1.jpg
✅ Crop 3: 'dog' (78.70%) → image2_20260502_152305_2.jpg

📦 Total crops saved: 3


#### Save to object_detection_metrics table

In [12]:
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, TimestampType

# filename     = os.path.basename(image_path)

filename_base = os.path.splitext(os.path.basename(image_path))[0]
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

detected     = len(detections) > 0
animal_count = len(detections)
timestamp    = datetime.now()

# image_name = f"{filename_base}_{timestamp_str}_{idx}.jpg"

# Build one row per detected object
if detections:
    rows = []
    for det in detections:
        for tag in det.get("tags", []):
            rows.append(Row(
                image_name    = f"{filename_base}_{timestamp_str}_{idx}.jpg",
                detected      = True,
                animal_count  = animal_count,
                object_name   = tag["name"],
                confidence    = float(tag["confidence"]),
                bbox_x        = int(det["boundingBox"]["x"]),
                bbox_y        = int(det["boundingBox"]["y"]),
                bbox_w        = int(det["boundingBox"]["w"]),
                bbox_h        = int(det["boundingBox"]["h"]),
                timestamp     = timestamp
            ))
else:
    # No animal detected — still log the transaction
    rows = [Row(
        image_name    = f"{filename_base}_{timestamp_str}_{idx}.jpg",
        detected      = False,
        animal_count  = 0,
        object_name   = "none",
        confidence    = 0.0,
        bbox_x        = 0,
        bbox_y        = 0,
        bbox_w        = 0,
        bbox_h        = 0,
        timestamp     = timestamp
    )]

detection_df = spark.createDataFrame(rows)
detection_df.write.mode("append").saveAsTable("object_detection_metrics")

print(f"✅ Detection metrics saved: {animal_count} animal(s) found in '{filename}'")

StatementMeta(, e4edba1f-148b-48a1-b3a8-b51aacbb2488, 14, Finished, Available, Finished, False)

NameError: name 'idx' is not defined